# 2 LeNet

[**LeNet (1998):**](https://en.wikipedia.org/wiki/LeNet) $\quad$ red **convolucional (CNN)** pionera, introducida en 1998 por Yann LeCun para MNIST

<div align="center"><img src="LeNet.png" width="300"/></div>

<div align="center"><img src="Figure_14.15.png" width="800"/></div>


<p style="page-break-after:always;"></p>


**Experimento con MNIST:** $\;$ con relus en lugar de sigmoides

In [1]:
import torch; import torch.nn as nn; import torch.nn.functional as F
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)
        self.fc1 = nn.Linear(16 * 4 * 4, 120); self.fc2 = nn.Linear(120, 84); self.fc3 = nn.Linear(84, 10)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), kernel_size=2)
        x = F.max_pool2d(F.relu(self.conv2(x)), kernel_size=2)
        x = torch.flatten(x, 1); x = F.relu(self.fc1(x)); x = F.relu(self.fc2(x)); x = self.fc3(x)
        return x
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
model = LeNet().to(device); print(device, model)

cuda LeNet(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=256, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


In [2]:
import numpy as np; import datasets
ds = datasets.load_dataset("ylecun/mnist").with_format("numpy"); ds
X_train = ds['train'][:]['image'].astype(np.float32).reshape(-1, 1, 28, 28) / 255.
y_train = ds['train'][:]['label'].astype(np.uint8)
X_test = ds['test'][:]['image'].astype(np.float32).reshape(-1, 1, 28, 28) / 255.
y_test = ds['test'][:]['label'].astype(np.uint8)
X = torch.Tensor(X_train).to(device); y = torch.Tensor(y_train).to(device, dtype=torch.long)
loss_fn = nn.CrossEntropyLoss(); optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)
model.train(); epochs = 200
for t in range(epochs):
    pred = model(X); loss = loss_fn(pred, y); loss.backward(); optimizer.step(); optimizer.zero_grad()
X = torch.Tensor(X_test).to(device); pred = model(X).detach().cpu().numpy()
acc = (pred.argmax(1) == y_test).sum().item() / y_test.size; print(f'Precisión {acc:.2%}')

Precisión 98.72%



<p style="page-break-after:always;"></p>
